[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-01-kestra-concepts.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Kestra Concepts — Declarative Orchestration Explained
**certified-journeys / kestra-certified** · Day 1 · Foundations

> **Goal for today:** Understand Kestra's core vocabulary — Flows, Tasks, Triggers, Executions, and Namespaces — and explain why its YAML-first design differs from code-first orchestrators like Airflow and Prefect.

In [ ]:
%pip install -q pyyaml requests

## Step 1 · What Is Kestra?

Kestra is an **event-driven, declarative workflow orchestrator**. Workflows (called *Flows*) are defined entirely in YAML — no Python decorators, no DAG objects, no SDK to import.

| Aspect | Kestra | Airflow | Prefect |
|--------|--------|---------|---------|
| Workflow definition | YAML | Python DAGs | Python `@flow` decorators |
| Versioning | YAML in Git | Python in Git | Python in Git |
| Non-developer friendly | ✅ Yes | ⚠ Limited | ⚠ Limited |
| Built-in UI | ✅ Yes | ✅ Yes | ✅ Yes |
| Plugin ecosystem | ✅ Large | ✅ Large | ✅ Growing |
| Multi-language tasks | ✅ Shell, Python, Node… | Python-centric | Python-centric |

Key insight: because Kestra flows are YAML, they are **data structures first** — parseable, lintable, and renderable by any language.

In [ ]:
import yaml

# A minimal Kestra flow represented as a Python dict
minimal_flow = {
    "id": "hello-kestra",
    "namespace": "tutorial",
    "tasks": [
        {
            "id": "greet",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Hello from Kestra!"
        }
    ]
}

# Render as the YAML you would paste into the Kestra UI
print(yaml.dump(minimal_flow, default_flow_style=False, sort_keys=False))

### What just happened?
- We modelled a Kestra flow as a plain Python dict — no Kestra library needed.
- **`id` + `namespace` uniquely identify a flow** across the entire Kestra server.
- `yaml.dump` produces valid YAML ready to paste into the Kestra UI or `kubectl apply`.
- The only required top-level keys are `id`, `namespace`, and `tasks`.

## Step 2 · Core Entities: Flows, Tasks, Triggers, Executions, Namespaces

Kestra's data model has five key entities:

| Entity | YAML key | Purpose |
|--------|----------|---------|
| **Flow** | root object | The complete workflow definition |
| **Task** | `tasks[].type` | A single unit of work (plugin call) |
| **Trigger** | `triggers[]` | What starts a flow (schedule, webhook, file event…) |
| **Execution** | runtime concept | One run of a Flow; has an ID, status, logs |
| **Namespace** | `namespace` | Hierarchical grouping, e.g. `company.team.project` |

Namespaces work like filesystem directories — they organise flows and control access.

In [ ]:
# Model all five entities together
full_example_flow = {
    "id": "daily-etl",
    "namespace": "company.data.engineering",    # hierarchical namespace
    "description": "Daily ETL pipeline — runs at midnight UTC",
    "labels": {
        "team": "data-engineering",
        "env": "production"
    },
    "triggers": [
        {
            "id": "daily-schedule",
            "type": "io.kestra.plugin.core.trigger.Schedule",
            "cron": "0 0 * * *"                 # midnight every day
        }
    ],
    "tasks": [
        {
            "id": "extract",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Extracting data from source…"
        },
        {
            "id": "transform",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Transforming data…"
        },
        {
            "id": "load",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Loading data to warehouse…"
        }
    ]
}

print("=== Full Flow YAML ===")
print(yaml.dump(full_example_flow, default_flow_style=False, sort_keys=False))
print(f"Entity summary:")
print(f"  Flow ID     : {full_example_flow['id']}")
print(f"  Namespace   : {full_example_flow['namespace']}")
print(f"  Tasks       : {[t['id'] for t in full_example_flow['tasks']]}")
print(f"  Triggers    : {[t['id'] for t in full_example_flow['triggers']]}")
print(f"  Labels      : {full_example_flow['labels']}")

### What just happened?
- **Labels** make flows discoverable in the Kestra UI — filter by team, environment, or any tag.
- The **Schedule trigger** is the most common — but Kestra also supports webhooks, file events, and flow-completion triggers.
- Each task runs **sequentially by default** — tasks execute in array order unless you use parallel or DAG task groups.
- An **Execution** is created each time the trigger fires; it tracks status (CREATED → RUNNING → SUCCESS/FAILED).

## Step 3 · Kestra Architecture Overview

Kestra's server has three main components:

```
┌────────────────────────────────────────────────┐
│                  Kestra Server                 │
│                                                │
│  ┌──────────┐  ┌──────────┐  ┌──────────────┐ │
│  │ Executor │  │ Scheduler│  │ Web UI / API │ │
│  │ (runs    │  │ (fires   │  │ (port 8080)  │ │
│  │  tasks)  │  │ triggers)│  │              │ │
│  └────┬─────┘  └────┬─────┘  └──────────────┘ │
│       │              │                          │
│  ┌────▼──────────────▼──────────────────────┐  │
│  │           Message Queue (Kafka / JDBC)   │  │
│  └───────────────────────────────────────────┘ │
│                                                │
│  ┌─────────────────────────────────────────┐   │
│  │    Repository (PostgreSQL / H2 / etc.)  │   │
│  └─────────────────────────────────────────┘   │
└────────────────────────────────────────────────┘
```

- **Executor** — picks up tasks from the queue and runs them using plugins
- **Scheduler** — checks cron expressions and fires trigger executions
- **Web UI / API** — where you create/edit flows, monitor executions, and view logs
- **Message Queue** — decouples execution from scheduling (Kafka for scale, JDBC for standalone)
- **Repository** — stores flow definitions and execution history

In [ ]:
# Simulate an Execution record — what Kestra stores when a flow runs
import datetime

def make_execution(flow_id: str, namespace: str, trigger_id: str = "manual") -> dict:
    """Build a mock Execution record in the shape Kestra uses."""
    now = datetime.datetime.utcnow()
    return {
        "id": f"exec-{now.strftime('%Y%m%d%H%M%S')}",
        "flowId": flow_id,
        "namespace": namespace,
        "state": {
            "current": "RUNNING",
            "histories": [
                {"state": "CREATED",  "date": (now - datetime.timedelta(seconds=2)).isoformat()},
                {"state": "RUNNING",  "date": now.isoformat()},
            ]
        },
        "trigger": {"type": "manual", "triggerId": trigger_id},
        "taskRunList": []
    }

execution = make_execution("daily-etl", "company.data.engineering")
print(yaml.dump(execution, default_flow_style=False, sort_keys=False))

### What just happened?
- An Execution tracks **state transitions** — every status change is appended to `histories`.
- The final state is one of: `CREATED`, `RUNNING`, `PAUSED`, `SUCCESS`, `FAILED`, `KILLED`, `WARNING`.
- **`taskRunList`** grows as each task runs — it holds per-task status, start/end time, and output references.
- The Kestra UI's **topology view** renders `taskRunList` as a visual DAG.

## Step 4 · Comparing YAML-first vs Code-first Orchestrators

Understanding the trade-offs helps you choose the right tool — and explain Kestra's design to teammates.

| Criterion | YAML-first (Kestra) | Code-first (Airflow / Prefect) |
|-----------|---------------------|--------------------------------|
| **Definition language** | YAML (declarative) | Python (imperative) |
| **Who can author flows** | Anyone with YAML knowledge | Python developers |
| **Version control** | Standard Git diff on YAML | Python file diff |
| **Dynamic logic** | Expressions (`{{ }}`) + plugins | Full Python |
| **Testing** | YAML validation + dry-run | Python unit tests |
| **IDE support** | JSON Schema autocomplete | Python type hints |
| **Multi-language** | Native (Shell, Python, Java…) | Python wrappers |

Kestra's expression language uses `{{ }}` for dynamic values:
- `{{ trigger.date }}` — the date the trigger fired
- `{{ outputs.taskId.value }}` — output from a previous task
- `{{ inputs.myParam }}` — a user-supplied input at execution time

In [ ]:
# Demonstrate Kestra's expression syntax via Python string rendering
# (In production Kestra evaluates {{ }} server-side with Pebble templating)

def render_kestra_expressions(flow_yaml: str, context: dict) -> str:
    """
    Simulate how Kestra resolves {{ expression }} placeholders.
    Real Kestra uses Pebble (Java) — this is a Python equivalent for learning.
    """
    import re
    def replace(match):
        key = match.group(1).strip()              # e.g. "trigger.date"
        parts = key.split(".")
        val = context
        try:
            for part in parts:
                val = val[part]
            return str(val)
        except (KeyError, TypeError):
            return match.group(0)                 # leave unresolved expression as-is
    return re.sub(r"\{\{\s*([^}]+)\s*\}\}", replace, flow_yaml)

# A flow with dynamic expressions
flow_with_expressions = """
id: parameterised-etl
namespace: tutorial
tasks:
  - id: log-date
    type: io.kestra.plugin.core.log.Log
    message: "Processing data for {{ trigger.date }}"
  - id: log-output
    type: io.kestra.plugin.core.log.Log
    message: "Previous task produced: {{ outputs.extract.value }}"
"""

# Simulate what Kestra fills in at runtime
runtime_context = {
    "trigger": {"date": "2026-06-07"},
    "outputs": {"extract": {"value": "42 rows loaded"}}
}

rendered = render_kestra_expressions(flow_with_expressions, runtime_context)
print("=== Flow with {{ }} expressions resolved ===")
print(rendered)

### What just happened?
- Kestra resolves `{{ }}` expressions at **execution time**, not when the flow is saved.
- This is how tasks pass data to downstream tasks: `{{ outputs.taskId.someKey }}`.
- **`trigger.date`** gives the exact timestamp the trigger fired — crucial for incremental loads.
- Our Python simulation matches what Kestra's Pebble engine does server-side.

## Step 5 · Installing Kestra with Docker

The quickest way to run Kestra locally is a single Docker command. We will generate the exact commands and a `docker-compose.yml` for a production-grade setup.

**Quick start (standalone, H2 in-memory DB):**
```bash
docker run --pull=always --rm \
  -p 8080:8080 \
  --user=root \
  -v /tmp/kestra-data:/app/storage \
  kestra/kestra:latest server local
```

After ~30 seconds the UI is at **http://localhost:8080**.

For production you need PostgreSQL + Kafka. Below we generate the docker-compose YAML.

In [ ]:
# Generate a production-grade Kestra docker-compose.yml
docker_compose = {
    "version": "3.8",
    "services": {
        "postgres": {
            "image": "postgres:15",
            "environment": {
                "POSTGRES_DB": "kestra",
                "POSTGRES_USER": "kestra",
                "POSTGRES_PASSWORD": "kestra"
            },
            "volumes": ["postgres-data:/var/lib/postgresql/data"],
            "healthcheck": {
                "test": ["CMD-SHELL", "pg_isready -U kestra"],
                "interval": "10s",
                "timeout": "5s",
                "retries": 5
            }
        },
        "kestra": {
            "image": "kestra/kestra:latest",
            "pull_policy": "always",
            "command": "server standalone",
            "user": "root",
            "depends_on": {"postgres": {"condition": "service_healthy"}},
            "volumes": [
                "kestra-data:/app/storage",
                "/var/run/docker.sock:/var/run/docker.sock",  # for Docker tasks
                "/tmp/kestra-wd:/tmp/kestra-wd"
            ],
            "ports": ["8080:8080", "8081:8081"],
            "environment": {
                "KESTRA_CONFIGURATION": """
datasources:
  postgres:
    url: jdbc:postgresql://postgres:5432/kestra
    username: kestra
    password: kestra
kestra:
  server:
    basic-auth:
      enabled: false
  repository:
    type: postgres
  storage:
    type: local
    local:
      base-path: /app/storage
  queue:
    type: postgres
  tasks:
    tmp-dir:
      path: /tmp/kestra-wd
""".strip()
            }
        }
    },
    "volumes": {
        "postgres-data": None,
        "kestra-data": None
    }
}

print(yaml.dump(docker_compose, default_flow_style=False, sort_keys=False, allow_unicode=True))

### What just happened?
- The standalone mode uses **PostgreSQL for both queue and repository** — no Kafka needed for smaller teams.
- Mounting `/var/run/docker.sock` lets Kestra spin up Docker containers as task workers (Docker tasks).
- `KESTRA_CONFIGURATION` is a YAML string embedded in the environment — Kestra reads it on startup.
- Ports: **8080** = Web UI + REST API; **8081** = internal management endpoint.

## Step 6 · Exploring the Kestra UI — Key Screens

Once Kestra is running at `http://localhost:8080`, here are the five screens to know:

| Screen | Path | What you see |
|--------|------|-------------|
| **Flows list** | `/ui/flows` | All flows, filterable by namespace/label |
| **Flow editor** | `/ui/flows/edit/<ns>/<id>` | YAML editor with syntax highlighting & validation |
| **Topology view** | `/ui/flows/<ns>/<id>/topology` | DAG of tasks and their connections |
| **Executions log** | `/ui/executions` | All past runs with status, duration, trigger |
| **Task logs** | `/ui/executions/<id>` | Per-task logs, outputs, and replay button |

Let's generate the URLs programmatically for our example flow:

In [ ]:
# Generate Kestra UI URLs for a given flow
def kestra_ui_urls(base_url: str, namespace: str, flow_id: str) -> dict:
    """Return the key Kestra UI screen URLs for a flow."""
    # Namespace separator in URLs is '.' but displayed hierarchically
    ns_encoded = namespace.replace(".", "%2E")  # Kestra encodes dots in URLs
    return {
        "flows_list":    f"{base_url}/ui/flows",
        "flow_editor":  f"{base_url}/ui/flows/edit/{namespace}/{flow_id}",
        "topology":     f"{base_url}/ui/flows/{namespace}/{flow_id}/topology",
        "executions":   f"{base_url}/ui/executions?namespace={namespace}&flowId={flow_id}",
        "api_trigger":  f"{base_url}/api/v1/executions/{namespace}/{flow_id}",
    }

urls = kestra_ui_urls(
    base_url="http://localhost:8080",
    namespace="company.data.engineering",
    flow_id="daily-etl"
)

print("Kestra UI URLs for 'daily-etl':")
for name, url in urls.items():
    print(f"  {name:<18} → {url}")

print()
print("Trigger a manual execution via curl:")
print(f"  curl -X POST {urls['api_trigger']}")

### What just happened?
- The Kestra REST API is available at the same host as the UI — every UI action is also an API call.
- **Topology view** is one of Kestra's best features — it auto-renders your task list as a DAG without any code.
- `POST /api/v1/executions/<namespace>/<flowId>` triggers a manual run — useful in CI/CD pipelines.
- The API also supports **creating and updating flows** — `PUT /api/v1/flows` with YAML body.

## Step 7 · Validating Flow Structure with Python

Before pasting YAML into Kestra, validate it with Python assertions. This is the YAML-first equivalent of type-checking your DAG code.

In [ ]:
def validate_kestra_flow(flow: dict) -> list[str]:
    """
    Validate a Kestra flow dict against common schema rules.
    Returns a list of error messages (empty list = valid).
    """
    errors = []

    # Required top-level fields
    for field in ["id", "namespace", "tasks"]:
        if field not in flow:
            errors.append(f"Missing required field: '{field}'")

    # id must be kebab-case (letters, digits, hyphens only)
    import re
    if "id" in flow and not re.match(r'^[a-z0-9][a-z0-9-]*$', flow["id"]):
        errors.append(f"Flow id '{flow['id']}' must be kebab-case (lowercase letters, digits, hyphens)")

    # namespace must use dots as separators
    if "namespace" in flow and " " in flow["namespace"]:
        errors.append("Namespace must not contain spaces — use dots as separators")

    # tasks must be a non-empty list
    if "tasks" in flow:
        if not isinstance(flow["tasks"], list) or len(flow["tasks"]) == 0:
            errors.append("'tasks' must be a non-empty list")
        else:
            seen_ids = set()
            for i, task in enumerate(flow["tasks"]):
                # Each task needs id and type
                for key in ["id", "type"]:
                    if key not in task:
                        errors.append(f"Task at index {i} is missing '{key}'")
                # Task ids must be unique within the flow
                if "id" in task:
                    if task["id"] in seen_ids:
                        errors.append(f"Duplicate task id: '{task['id']}'")
                    seen_ids.add(task["id"])

    return errors


# Test with a valid flow
print("Valid flow errors:", validate_kestra_flow(full_example_flow))

# Test with an invalid flow
bad_flow = {
    "id": "My Bad Flow",       # spaces not allowed
    "namespace": "tutorial",
    "tasks": [
        {"id": "step1", "type": "io.kestra.plugin.core.log.Log"},
        {"id": "step1", "type": "io.kestra.plugin.core.log.Log"},  # duplicate id
        {"type": "io.kestra.plugin.core.log.Log"},                  # missing id
    ]
}
print("\nBad flow errors:")
for err in validate_kestra_flow(bad_flow):
    print(f"  ❌ {err}")

### What just happened?
- We built a lightweight validator that catches the most common flow authoring mistakes.
- **Kebab-case IDs** are enforced by Kestra — spaces and uppercase letters will be rejected on import.
- **Unique task IDs** matter because output references (`{{ outputs.taskId.value }}`) use the ID as the key.
- Kestra also has a JSON Schema you can point a YAML Language Server at for IDE autocomplete.

In [ ]:
# Challenge: Build a valid Kestra flow YAML for a weekly report pipeline
#
# Requirements:
#   - Flow id: "weekly-report"
#   - Namespace: "analytics.reports"
#   - Labels: team="analytics", env="production"
#   - A Schedule trigger that fires every Monday at 08:00 UTC
#   - Three tasks in order:
#       1. fetch-data  — Log task with message "Fetching report data for {{ trigger.date }}"
#       2. build-report — Log task with message "Building report…"
#       3. send-email  — Log task with message "Sending report to stakeholders"
#   - Validate the flow with validate_kestra_flow()
#   - Print the YAML

# Your solution here:
weekly_report_flow = {
    "id": "weekly-report",
    # TODO: add namespace, labels, triggers, and tasks
}

errors = validate_kestra_flow(weekly_report_flow)
if errors:
    print("Validation errors:", errors)
else:
    print(yaml.dump(weekly_report_flow, default_flow_style=False, sort_keys=False))

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Flow | YAML document; uniquely identified by `id` + `namespace` |
| Namespace | Hierarchical grouping using dots, e.g. `company.team.project` |
| Task | One unit of work; must have `id` and `type` (plugin class) |
| Trigger | What starts a flow: Schedule, Webhook, File event, Flow completion |
| Execution | One runtime instance of a Flow; tracks status and task outputs |
| Expression | `{{ }}` syntax for dynamic values resolved at execution time |
| YAML-first | Declarative, diffable, Git-friendly — no Python SDK needed |
| Docker install | `docker run kestra/kestra:latest server local` for local dev |

> **Tip:** Kestra's YAML-first design means your entire workflow is version-controlled, reviewable in a PR, and executable by anyone who can run Docker.

---
## What's next
**Day 2** → Write your first real Kestra flow — YAML structure deep-dive, required fields, chaining tasks, and importing/exporting flows between namespaces.

Mark Day 1 complete in your [tracker](../index.html).